# 第 2 讲：资源核算

这是 CS336 Lecture 02 的可执行中文学习版。

- 官方来源：`external/cs336-lectures/lecture_02.py`
- 官方形式：executable lecture
- 英文标题：Resource Accounting

这份 notebook 是学习版，不是逐字翻译。它按官方讲义的结构和节奏整理主线，并在适合的位置加入小实验。


## 0. 本讲你要抓住什么

这一讲把训练语言模型拆成 compute、memory、dtype、FLOPs、arithmetic intensity 和训练循环的资源账。目标不是背公式，而是养成先估算资源再运行实验的习惯。

学习方式：

1. 先读每节的中文解释。
2. 运行对应代码 cell。
3. 改一个参数，观察输出如何变化。
4. 最后用检查点问题自测。


## 1. 本讲路线图

从 tensor 和 dtype 出发，估算 memory；从矩阵乘法出发，估算 FLOPs；再把这些账接到 forward、backward、optimizer 和训练循环。


In [1]:
lecture = 2
title = 'Resource Accounting'
source = 'external/cs336-lectures/lecture_02.py'
print(f"Lecture {lecture}: {title}")
print("source:", source)


Lecture 2: Resource Accounting
source: external/cs336-lectures/lecture_02.py


## 2. Tensor 是所有状态的容器

数据、参数、activation、gradient、optimizer state 都是 tensor。每个 tensor 的内存由元素个数和 dtype 决定。


## 3. 浮点格式和显存

fp32 通常 4 bytes，fp16/bf16 通常 2 bytes。混合精度训练的核心收益之一是降低显存和带宽压力。


## 4. FLOPs 和训练时间

常用粗略估计：训练 N 个参数、D 个 token 的 dense Transformer，大约需要 6ND FLOPs。


## 5. Arithmetic intensity

FLOPs / bytes moved。强度高的操作更可能 compute-bound，强度低的操作更可能 memory-bound。


## 6. 训练循环的资源

参数、梯度、optimizer state、activation 都要占内存；backward 和 optimizer step 也要消耗 compute。


## 动手实验 1：显存账：dtype 与 tensor size

先直接运行，再改一个数字或字符串。


In [2]:
def tensor_bytes(shape, bytes_per_value):
    n = 1
    for dim in shape:
        n *= dim
    return n * bytes_per_value

shape = (32, 2048, 4096)
for dtype, b in [("fp32", 4), ("fp16/bf16", 2), ("fp8", 1)]:
    gb = tensor_bytes(shape, b) / 1e9
    print(dtype, f"{gb:.2f} GB")


fp32 1.07 GB
fp16/bf16 0.54 GB
fp8 0.27 GB


## 动手实验 2：训练 FLOPs 粗算

先直接运行，再改一个数字或字符串。


In [3]:
def training_flops(num_params, num_tokens):
    return 6 * num_params * num_tokens

params = 7e9
tokens = 1e12
flops = training_flops(params, tokens)
h100_bf16_flops = 989e12  # rough effective order, not a promise
gpus = 128
mfu = 0.4
seconds = flops / (h100_bf16_flops * gpus * mfu)
print("total FLOPs:", f"{flops:.2e}")
print("rough days:", seconds / 86400)


total FLOPs: 4.20e+22
rough days: 9.599957167733962


## 动手实验 3：Arithmetic intensity 玩具例子

先直接运行，再改一个数字或字符串。


In [4]:
def intensity(flops, bytes_moved):
    return flops / bytes_moved

ops = {
    "elementwise add": (1_000_000, 12_000_000),
    "matmul-ish": (2_000_000_000, 32_000_000),
}
for name, (flops, bytes_moved) in ops.items():
    print(name, "intensity=", intensity(flops, bytes_moved))


elementwise add intensity= 0.08333333333333333
matmul-ish intensity= 62.5


## 今日检查点

1. 能解释为什么 dtype 会直接改变显存。
2. 能用 6ND 粗估训练 FLOPs。
3. 能区分 compute-bound 和 memory-bound 的直觉。
4. 能说出训练时除了参数外还有哪些东西占显存。

如果这些能讲清楚，这一讲的第一轮学习就完成了。
